In [4]:
import os
import requests
from concurrent.futures import ThreadPoolExecutor, as_completed

DATASET = "hfmlsoc/hub_weekly_snapshots"
API_URL = f"https://datasets-server.huggingface.co/parquet?dataset={DATASET}"
OUT_DIR = "data/raw/hf_snapshots/default_train"
MAX_WORKERS = 12
CHUNK_SIZE = 8 * 1024 * 1024  # 8 MB

def get_parquet_urls():
    r = requests.get(API_URL, timeout=60)
    r.raise_for_status()
    payload = r.json()

    parquet_files = payload.get("parquet_files", [])
    if not parquet_files:
        raise RuntimeError("No parquet_files found.")

    urls = []
    for item in parquet_files:
        if item.get("config") == "default" and item.get("split") == "train":
            urls.append((item["filename"], item["url"], item.get("size")))

    if not urls:
        raise RuntimeError("No parquet shards found for default/train.")

    return urls

def download_one(file_info):
    filename, url, expected_size = file_info
    out_file = os.path.join(OUT_DIR, filename)

    if os.path.exists(out_file):
        local_size = os.path.getsize(out_file)
        if expected_size is None or local_size == expected_size:
            return f"SKIP  {filename}"
        else:
            os.remove(out_file)

    session = requests.Session()
    session.headers.update({"User-Agent": "hf-snapshot-downloader/0.1"})

    with session.get(url, stream=True, timeout=120) as r:
        r.raise_for_status()
        with open(out_file, "wb") as f:
            for chunk in r.iter_content(chunk_size=CHUNK_SIZE):
                if chunk:
                    f.write(chunk)

    return f"DONE  {filename}"

def main():
    os.makedirs(OUT_DIR, exist_ok=True)
    parquet_urls = get_parquet_urls()

    print(f"Found {len(parquet_urls)} parquet shards.")
    print(f"Downloading with {MAX_WORKERS} workers...")

    done = 0
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = [executor.submit(download_one, item) for item in parquet_urls]

        for future in as_completed(futures):
            done += 1
            try:
                msg = future.result()
                print(f"[{done}/{len(parquet_urls)}] {msg}")
            except Exception as e:
                print(f"[{done}/{len(parquet_urls)}] ERROR: {e}")

    print("Download completed.")

if __name__ == "__main__":
    main()

Found 339 parquet shards.
[1/339] SKIP  0000.parquet
[2/339] SKIP  0008.parquet
[3/339] SKIP  0010.parquet
[4/339] SKIP  0011.parquet
[5/339] SKIP  0005.parquet
[6/339] SKIP  0002.parquet
[7/339] SKIP  0004.parquet
[8/339] SKIP  0003.parquet
[9/339] SKIP  0007.parquet
[10/339] SKIP  0009.parquet
[11/339] SKIP  0001.parquet
[12/339] SKIP  0006.parquet
[13/339] DONE  0021.parquet
[14/339] DONE  0012.parquet
[15/339] DONE  0024.parquet
[16/339] DONE  0025.parquet
[17/339] DONE  0016.parquet
[18/339] DONE  0027.parquet
[19/339] DONE  0013.parquet
[20/339] DONE  0026.parquet
[21/339] DONE  0015.parquet
[22/339] DONE  0014.parquet
[23/339] DONE  0023.parquet
[24/339] DONE  0029.parquet
[25/339] DONE  0017.parquet
[26/339] DONE  0018.parquet
[27/339] DONE  0028.parquet
[28/339] DONE  0022.parquet
[29/339] DONE  0020.parquet
[30/339] DONE  0019.parquet
[31/339] DONE  0034.parquet
[32/339] DONE  0030.parquet
[33/339] DONE  0038.parquet
[34/339] DONE  0037.parquet
[35/339] DONE  0039.parquet
[36

KeyboardInterrupt: 